In [20]:
from google.cloud import bigquery
from datetime import date, datetime, timedelta, timezone
import pandas as pd
import yaml
from pathlib import Path

In [21]:
ruta = Path(r"C:\Users\omen\OneDrive\Documentos\Proyectos\olist_ecomm_analytics\ecomm_analytics_project\Ingestion\properties.yaml")

with ruta.open("r", encoding="utf-8") as f:
    datos = yaml.safe_load(f)

print(datos)

{'project': {'project': 'inbound-byte-417306', 'schema': 'raw_ecomm_analytics'}}


In [22]:
proj_ingest_path = Path.cwd()
print(proj_ingest_path)


c:\Users\omen\OneDrive\Documentos\Proyectos\olist_ecomm_analytics\ecomm_analytics_project\Ingestion


In [23]:
client = bigquery.Client(project=datos["project"]["project"])

execution_ts_utc = datetime.now(timezone.utc)


In [24]:
def load_raw_customers(datasource):

    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_customers"
    df = pd.DataFrame(datasource)
    df = df[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "customer_id": "string",
                    "customer_unique_id": "string",
                    "customer_zip_code_prefix": "string",
                    "customer_city": "string",
                    "customer_state": "string",
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })


    try:    
        create_cust_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                customer_id STRING,
                customer_unique_id STRING,
                customer_zip_code_prefix STRING,
                customer_city STRING,
                customer_state STRING,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_cust_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating raw_customers table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()
    
        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise

In [25]:
def load_raw_geolocation(datasource):

    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_geolocation"
    df = pd.DataFrame(datasource)
    df = df[["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "geolocation_zip_code_prefix": "string",
                    "geolocation_lat": "float64",
                    "geolocation_lng": "float64",
                    "geolocation_city": "string",
                    "geolocation_state": "string",
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })


    try:    
        create_geolocation_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                geolocation_zip_code_prefix STRING,
                geolocation_lat FLOAT64,
                geolocation_lng FLOAT64,
                geolocation_city STRING,
                geolocation_state STRING,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_geolocation_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating raw_geolocation table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()
    
        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise

In [26]:
def load_raw_order_items(datasource): 

    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_order_items"
    df = pd.DataFrame(datasource)
    df = df[["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "order_id": "string",
                    "order_item_id": "int64",
                    "product_id": "string",
                    "seller_id": "string",
                    "shipping_limit_date": "datetime64[ns]",
                    "price": "float64",
                    "freight_value": "float64", 
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })


    try:    
        create_order_items_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                order_id STRING,
                order_item_id INTEGER,
                product_id STRING,
                seller_id STRING,
                shipping_limit_date TIMESTAMP,
                price FLOAT64,
                freight_value FLOAT64,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_order_items_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating raw_order_items table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()
    
        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise


In [27]:
def load_raw_orders(datasource): 
    
    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_orders"
    df = pd.DataFrame(datasource)
    df = df[["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "order_id": "string",
                    "customer_id": "string",
                    "order_status": "string",
                    "order_purchase_timestamp": "datetime64[ns]",
                    "order_approved_at": "datetime64[ns]",
                    "order_delivered_carrier_date": "datetime64[ns]",
                    "order_delivered_customer_date": "datetime64[ns]",
                    "order_estimated_delivery_date": "datetime64[ns]",
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })


    try:    
        create_order_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                order_id STRING,
                customer_id STRING,
                order_status STRING,
                order_purchase_timestamp STRING,
                order_approved_at TIMESTAMP,
                order_delivered_carrier_date TIMESTAMP,
                order_delivered_customer_date TIMESTAMP,
                order_estimated_delivery_date TIMESTAMP,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_order_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating orders table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()
    
        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise


In [28]:

def load_raw_order_payments(datasource): 
    
    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_order_payments"
    df = pd.DataFrame(datasource)
    df = df[["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "order_id": "string",
                    "payment_sequential": "int64",
                    "payment_type": "string",
                    "payment_installments": "int64",
                    "payment_value": "float64", 
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })


    try:    
        create_order_payments_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                order_id STRING,
                payment_sequential INT64,
                payment_type STRING,
                payment_installments INT64,
                payment_value FLOAT64,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_order_payments_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating order payments table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()
    
        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise

In [29]:

def load_raw_order_reviews(datasource): 
   
    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_order_reviews"
    df = pd.DataFrame(datasource)
    df = df[["review_id", "order_id", "review_score", "review_comment_title", "review_comment_message", "review_creation_date", "review_answer_timestamp"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "review_id": "string",
                    "order_id": "string",
                    "review_score": "int64",
                    "review_comment_title": "string",
                    "review_comment_message": "string",
                    "review_creation_date": "datetime64[ns]",
                    "review_answer_timestamp": "datetime64[ns]",
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })


    try:    
        create_order_reviews_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                review_id STRING,
                order_id STRING,
                review_score INT64,
                review_comment_title STRING,
                review_comment_message STRING,
                review_creation_date TIMESTAMP,
                review_answer_timestamp TIMESTAMP,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_order_reviews_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating order reviews table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()
    
        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise

In [30]:

def load_raw_category_catalog(datasource): 

    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_product_category_name_translation"
    df = pd.DataFrame(datasource) 
    df = df[["product_category_name", "product_category_name_english"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "product_category_name": "string",
                    "product_category_name_english": "string",
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })

    try:    
        create_product_category_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                product_category_name STRING,
                product_category_name_english STRING,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_product_category_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating product category table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()

        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise

In [31]:

def load_raw_sellers(datasource):  

    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_sellers"
    df = pd.DataFrame(datasource)
    df = df[["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "seller_id": "string",
                    "seller_zip_code_prefix": "string",
                    "seller_city": "string",
                    "seller_state": "string",
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })


    try:    
        create_sellers_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                seller_id STRING,
                seller_zip_code_prefix STRING,
                seller_city STRING,
                seller_state STRING,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_sellers_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating sellers table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()
    
        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise

In [32]:

def load_raw_products(datasource):

    target_table = f"{datos['project']['project']}.{datos['project']['schema']}.raw_products"
    df = pd.DataFrame(datasource)
    df = df[["product_id", "product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]].copy()
    df["loaded_ts_utc"] = execution_ts_utc
    df = df.astype({
                    "product_id": "string",
                    "product_category_name": "string",
                    "product_name_lenght": "string",
                    "product_description_lenght": "string",
                    "product_photos_qty": "string",
                    "product_weight_g": "string",
                    "product_length_cm": "string",
                    "product_height_cm": "string",
                    "product_width_cm": "string", 
                    "loaded_ts_utc": "datetime64[ns, UTC]"
                    })

    try:    
        create_products_table = f"""
            CREATE OR REPLACE TABLE `{target_table}`
            (
                product_id STRING,
                product_category_name STRING,
                product_name_lenght STRING,
                product_description_lenght STRING,
                product_photos_qty STRING,
                product_weight_g STRING,
                product_length_cm STRING,
                product_height_cm STRING,
                product_width_cm STRING,
                loaded_ts_utc TIMESTAMP
            )
        """
        query_job = client.query(create_products_table)
        query_job.result()

    except Exception as e:
        print(f"Error occurred while creating products table: {e}")
        raise


    try: 
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
            )
        
        job = client.load_table_from_dataframe(df, target_table, job_config=job_config)
        job.result()

        target_rows = client.get_table(target_table).num_rows
        print(f"Information loaded to {target_table} table successfully with {target_rows} rows.")

    except Exception as e:
        print(f"Error loading information to table: {e}")
        raise

In [33]:
raw_tables = {"raw_customers": {"source":"olist_customers_dataset",
                                "function": load_raw_customers
                                    }, 
              "raw_geolocation": {"source":'olist_geolocation_dataset',
                                  "function": load_raw_geolocation
                                    }, 
              "raw_order_items": {"source":'olist_order_items_dataset',
                                  "function": load_raw_order_items
                                    }, 
              "raw_orders": {"source":'olist_orders_dataset', 
                            "function": load_raw_orders},
              "raw_order_payments": {"source":'olist_order_payments_dataset', 
                                     "function": load_raw_order_payments},
              "raw_order_reviews": {"source":'olist_order_reviews_dataset', 
                                    "function": load_raw_order_reviews},
              "raw_product_category_name_translation": {"source":'product_category_name_translation', 
                                                        "function": load_raw_category_catalog},
              "raw_sellers": {"source":'olist_sellers_dataset', 
                              "function": load_raw_sellers},
              "raw_products": {"source":'olist_products_dataset', 
                               "function": load_raw_products}
            }

print(len(raw_tables.keys()))

9


In [34]:
selected_tables = ["raw_products"]
filtered_raw_tables = {
    table: config
    for table, config in raw_tables.items()
    if table in selected_tables
    }

print(filtered_raw_tables)


{'raw_products': {'source': 'olist_products_dataset', 'function': <function load_raw_products at 0x000002A263530FE0>}}


In [35]:
for i, (raw_table_name, value) in enumerate(raw_tables.items()):

    try:
        print(f"Processing table {i + 1}/{len(raw_tables)}: {raw_table_name}")
        print(f"Extracted from CSV: {value['source']}")

        raw_path = f"{proj_ingest_path}/{value['source']}.csv" 

        df = pd.read_csv(raw_path, encoding="utf-8-sig") 
        csv_rows = df.shape[0]  

        print(f"Extracted {raw_table_name} successfully. Count: {csv_rows}")

        create_table_function = value.get("function")

        if create_table_function is None:
            print(f"No function defined for {raw_table_name}. Skipping.")
            print("")
            continue
 
        create_table_query = create_table_function(df)

        target_table_id = f"{datos['project']['project']}.{datos['project']['schema']}.{raw_table_name}" 
        table_loaded_rows = client.get_table(target_table_id).num_rows 

        if table_loaded_rows == csv_rows:
            print(f"BigQuery table {raw_table_name} match in rows vs CSV.")  
        else:
            print(f"Row count mismatch for {raw_table_name}: CSV has {csv_rows} rows, but BigQuery has {table_loaded_rows} rows.")

    except Exception as e:
        print(f"Error occurred while processing {raw_table_name}: {e}")

    print("")

Processing table 1/9: raw_customers
Extracted from CSV: olist_customers_dataset
Extracted raw_customers successfully. Count: 99441
Information loaded to inbound-byte-417306.raw_ecomm_analytics.raw_customers table successfully with 99441 rows.
BigQuery table raw_customers match in rows vs CSV.

Processing table 2/9: raw_geolocation
Extracted from CSV: olist_geolocation_dataset
Extracted raw_geolocation successfully. Count: 1000163
Information loaded to inbound-byte-417306.raw_ecomm_analytics.raw_geolocation table successfully with 1000163 rows.
BigQuery table raw_geolocation match in rows vs CSV.

Processing table 3/9: raw_order_items
Extracted from CSV: olist_order_items_dataset
Extracted raw_order_items successfully. Count: 112650
Information loaded to inbound-byte-417306.raw_ecomm_analytics.raw_order_items table successfully with 112650 rows.
BigQuery table raw_order_items match in rows vs CSV.

Processing table 4/9: raw_orders
Extracted from CSV: olist_orders_dataset
Extracted raw_o